# SignScribe - LLM Fine-Tuning (+ Optional Classifier Training)

**Run this notebook on Google Colab with a T4 GPU runtime.**

This notebook:
1. (Optional) Trains the BiLSTM gesture classifier on your recorded data (~5-10 min on T4)
2. Fine-tunes T5-small with LoRA on gloss-to-sentence pairs (~15-30 min on T4)
3. Exports the trained weights for download to your local machine

## Setup
1. Upload this notebook to Google Colab
2. Go to **Runtime > Change runtime type > T4 GPU**
3. Upload your data (see cells below for instructions)
4. Run all cells in order

## Cell 1: Install Dependencies

In [ ]:
!pip install -q torch transformers peft datasets sentencepiece accelerate scikit-learn tqdm

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## Cell 2: Upload Your Data

**Option A (recommended): Upload a zip of your `data/raw/` folder**

On your local machine:
```
cd c:\Users\ratna\signscribe
python -c "import shutil; shutil.make_archive('data_raw', 'zip', 'data', 'raw')"
```
Then upload `data_raw.zip` when prompted below.

**Option B: Mount Google Drive** (if you've synced the data there)

In [ ]:
import os
import zipfile
from google.colab import files

# Choose your upload method:
UPLOAD_METHOD = "zip"  # Change to "drive" if using Google Drive

if UPLOAD_METHOD == "zip":
    print("Upload your data_raw.zip file:")
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    os.makedirs("data/raw", exist_ok=True)
    with zipfile.ZipFile(zip_name, 'r') as z:
        z.extractall("data/raw")
    print(f"Extracted to data/raw/")

elif UPLOAD_METHOD == "drive":
    from google.colab import drive
    drive.mount('/content/drive')
    # Adjust this path to where your data is on Drive:
    DRIVE_DATA_PATH = "/content/drive/MyDrive/signscribe/data/raw"
    os.makedirs("data/raw", exist_ok=True)
    !cp -r {DRIVE_DATA_PATH}/* data/raw/
    print(f"Copied from Drive to data/raw/")

# Verify
for label_dir in sorted(os.listdir("data/raw")):
    if os.path.isdir(f"data/raw/{label_dir}"):
        count = len([f for f in os.listdir(f"data/raw/{label_dir}") if f.endswith('.npy')])
        print(f"  {label_dir}: {count} samples")

---
## (Optional) Section A: Train the BiLSTM Classifier on GPU

This is ~10-20x faster than CPU training on your laptop.
Skip this section if you've already trained locally.

In [ ]:
# ===== BiLSTM Classifier - Config =====

# Match your local src/config.py values
NUM_LANDMARKS = 21
NUM_COORDS = 3
NUM_HANDS = 2
FEATURE_DIM = NUM_LANDMARKS * NUM_COORDS * NUM_HANDS  # 126
WINDOW_SIZE = 30
LSTM_HIDDEN_SIZE = 128
LSTM_NUM_LAYERS = 2
LSTM_BIDIRECTIONAL = True
LSTM_DROPOUT = 0.3

TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

print("Config loaded.")
print(f"  Feature dim: {FEATURE_DIM}")
print(f"  Window size: {WINDOW_SIZE}")

In [ ]:
# ===== BiLSTM Classifier - Model Definition =====

import torch
import torch.nn as nn

class GestureClassifier(nn.Module):
    def __init__(self, input_dim=FEATURE_DIM, hidden_size=LSTM_HIDDEN_SIZE,
                 num_layers=LSTM_NUM_LAYERS, num_classes=96,
                 dropout=LSTM_DROPOUT, bidirectional=LSTM_BIDIRECTIONAL):
        super().__init__()
        self.num_directions = 2 if bidirectional else 1
        self.input_norm = nn.LayerNorm(input_dim)
        self.lstm = nn.LSTM(
            input_size=input_dim, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size * self.num_directions, num_classes),
        )

    def forward(self, x):
        x = self.input_norm(x)
        _, (h_n, _) = self.lstm(x)
        if self.num_directions == 2:
            h_final = torch.cat([h_n[-2], h_n[-1]], dim=1)
        else:
            h_final = h_n[-1]
        return self.classifier(h_final)

print("Model class defined.")

In [ ]:
# ===== BiLSTM Classifier - Dataset + Training =====

import numpy as np
import random
import time
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import accuracy_score, classification_report

DATA_DIR = Path("data/raw")

class SimpleGestureDataset(Dataset):
    def __init__(self, files_and_labels, augment=False, aug_factor=4):
        self.samples = files_and_labels
        self.augment = augment
        self.aug_factor = aug_factor

    def __len__(self):
        return len(self.samples) * (1 + self.aug_factor) if self.augment else len(self.samples)

    def __getitem__(self, idx):
        n = len(self.samples)
        orig_idx = idx % n
        variant = idx // n
        fpath, label_idx = self.samples[orig_idx]
        w = np.load(fpath).astype(np.float32)
        if variant == 1:  # mirror
            for hand in range(NUM_HANDS):
                off = hand * NUM_LANDMARKS * NUM_COORDS
                for lm in range(NUM_LANDMARKS):
                    w[:, off + lm * NUM_COORDS] = 1.0 - w[:, off + lm * NUM_COORDS]
        elif variant == 2:  # scale
            s = random.uniform(0.85, 1.15)
            mask = w != 0
            w[mask] *= s
        elif variant == 3:  # temporal shift
            shift = random.randint(-3, 3)
            if shift > 0: w[shift:] = w[:-shift]; w[:shift] = 0
            elif shift < 0: w[:shift] = w[-shift:]; w[shift:] = 0
        elif variant == 4:  # noise
            mask = w != 0
            w[mask] += np.random.normal(0, 0.01, w[mask].shape).astype(np.float32)
        return torch.from_numpy(w), label_idx

# Discover data
label_dirs = sorted(d for d in DATA_DIR.iterdir() if d.is_dir())
label_names = [d.name for d in label_dirs]
label_to_idx = {name: i for i, name in enumerate(label_names)}
all_samples = []
for ld in label_dirs:
    idx = label_to_idx[ld.name]
    for f in sorted(ld.glob('*.npy')):
        all_samples.append((f, idx))

print(f"Found {len(all_samples)} samples across {len(label_names)} classes")
print(f"Labels: {label_names}")

num_classes = len(label_names)
n_total = len(all_samples)
n_train = int(n_total * TRAIN_RATIO)
n_val = int(n_total * VAL_RATIO)
n_test = n_total - n_train - n_val
if n_val == 0 and n_total >= 3: n_val = 1; n_train -= 1
if n_test == 0 and n_total >= 3: n_test = 1; n_train -= 1

random.seed(42)
random.shuffle(all_samples)
train_samples = all_samples[:n_train]
val_samples = all_samples[n_train:n_train+n_val]
test_samples = all_samples[n_train+n_val:]

train_ds = SimpleGestureDataset(train_samples, augment=True)
val_ds = SimpleGestureDataset(val_samples)
test_ds = SimpleGestureDataset(test_samples)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=32)
test_dl = DataLoader(test_ds, batch_size=32)

print(f"Split: {n_train} train / {n_val} val / {n_test} test")
print(f"Effective train samples (with aug): {len(train_ds)}")

# Train
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GestureClassifier(num_classes=num_classes).to(device)
param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel parameters: {param_count:,}")
print(f"Training on: {device}")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

EPOCHS = 50
PATIENCE = 10
best_val_loss = float('inf')
patience_counter = 0
start = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for bx, by in train_dl:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        logits = model(bx)
        loss = criterion(logits, by)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * bx.size(0)
        correct += (logits.argmax(1) == by).sum().item()
        total += bx.size(0)

    train_loss = total_loss / total
    train_acc = correct / total

    # Validation
    model.eval()
    val_loss = 0; val_correct = 0; val_total = 0
    with torch.no_grad():
        for bx, by in val_dl:
            bx, by = bx.to(device), by.to(device)
            logits = model(bx)
            val_loss += criterion(logits, by).item() * bx.size(0)
            val_correct += (logits.argmax(1) == by).sum().item()
            val_total += bx.size(0)
    val_loss /= max(val_total, 1)
    val_acc = val_correct / max(val_total, 1)
    scheduler.step(val_loss)

    if epoch % 5 == 0 or epoch == 1:
        print(f"  Epoch {epoch:3d}  |  Train: {train_loss:.4f} / {train_acc:.4f}  |  Val: {val_loss:.4f} / {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save({
            'model_state_dict': model.state_dict(),
            'label_names': label_names,
            'num_classes': num_classes,
            'epoch': epoch,
            'val_loss': val_loss,
            'val_acc': val_acc,
        }, 'classifier.pt')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"  Early stopping at epoch {epoch}")
            break

elapsed = time.time() - start
print(f"\nTraining complete in {elapsed:.1f}s ({elapsed/60:.1f} min)")

# Test evaluation
ckpt = torch.load('classifier.pt', weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for bx, by in test_dl:
        bx = bx.to(device)
        preds = model(bx).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(by.numpy())

print(f"\nTest Accuracy: {accuracy_score(all_labels, all_preds):.4f}")
if len(set(all_labels)) > 1:
    print(classification_report(all_labels, all_preds,
          target_names=[label_names[i] for i in sorted(set(all_labels + all_preds))],
          zero_division=0))

In [ ]:
# ===== Download Classifier Weights =====
# Run this cell to download classifier.pt to your local machine.
# Then place it at: models/classifier.pt

from google.colab import files
files.download('classifier.pt')
print("Download started. Place the file at: models/classifier.pt")

---
## Section B: Fine-Tune T5-small with LoRA

This section fine-tunes T5-small on gloss-to-sentence pairs using
LoRA (Low-Rank Adaptation) via Hugging Face's PEFT library.

### About the Training Data

You need a CSV file with two columns:
- `gloss`: space-separated ISL gloss tokens (e.g., `I SCHOOL GO`)
- `sentence`: the target English sentence (e.g., `I go to school.`)

**If you don't have real pairs yet**, the next cell generates a
synthetic training set from templates. Review and edit the
generated CSV before training.

In [ ]:
# ===== Generate Synthetic Gloss-Sentence Training Data =====
# This creates template-based training pairs.
# REVIEW the output CSV and edit/add pairs before training!

import csv

SYNTHETIC_PAIRS = [
    # Basic greetings
    ("HELLO", "Hello."),
    ("GOODBYE", "Goodbye."),
    ("THANK_YOU", "Thank you."),
    ("SORRY", "I am sorry."),
    ("PLEASE", "Please."),
    ("WELCOME", "You are welcome."),
    ("GOOD_MORNING", "Good morning."),
    ("GOOD_NIGHT", "Good night."),
    ("YES", "Yes."),
    ("NO", "No."),
    
    # Subject + Verb patterns
    ("I GO", "I go."),
    ("I COME", "I come."),
    ("I EAT", "I eat."),
    ("I DRINK", "I drink."),
    ("I SLEEP", "I sleep."),
    ("I WORK", "I work."),
    ("I LEARN", "I learn."),
    ("I PLAY", "I play."),
    ("YOU GO", "You go."),
    ("YOU COME", "You come."),
    ("HE GO", "He goes."),
    ("SHE GO", "She goes."),
    ("WE GO", "We go."),
    ("THEY GO", "They go."),
    
    # Subject + Object + Verb (SOV -> SVO)
    ("I SCHOOL GO", "I go to school."),
    ("I HOME GO", "I go home."),
    ("I HOSPITAL GO", "I go to the hospital."),
    ("I MARKET GO", "I go to the market."),
    ("I OFFICE GO", "I go to the office."),
    ("I FOOD EAT", "I eat food."),
    ("I WATER DRINK", "I drink water."),
    ("I BOOK READ", "I read a book."),
    ("I FRIEND HELP", "I help my friend."),
    ("I TEACHER ASK", "I ask the teacher."),
    ("HE SCHOOL GO", "He goes to school."),
    ("SHE HOSPITAL GO", "She goes to the hospital."),
    ("HE FOOD EAT", "He eats food."),
    ("SHE BOOK READ", "She reads a book."),
    ("WE PARK GO", "We go to the park."),
    ("THEY MARKET GO", "They go to the market."),
    ("YOU WATER DRINK", "You drink water."),
    ("I NAME WRITE", "I write my name."),
    ("HE TEACHER ASK", "He asks the teacher."),
    ("SHE FRIEND CALL", "She calls her friend."),
    
    # With tense markers
    ("I SCHOOL GO PAST", "I went to school."),
    ("I SCHOOL GO YESTERDAY", "I went to school yesterday."),
    ("I SCHOOL GO FUTURE", "I will go to school."),
    ("I SCHOOL GO TOMORROW", "I will go to school tomorrow."),
    ("HE HOSPITAL GO PAST", "He went to the hospital."),
    ("SHE FOOD EAT PAST", "She ate food."),
    ("I BOOK READ PAST", "I read a book."),
    ("WE PARK GO FUTURE", "We will go to the park."),
    ("THEY MARKET GO YESTERDAY", "They went to the market yesterday."),
    ("I WATER DRINK FUTURE", "I will drink water."),
    
    # Questions (using question mark)
    ("YOU HELP WANT", "Do you want help?"),
    ("YOU SCHOOL GO", "Do you go to school?"),
    ("YOU NAME WHAT", "What is your name?"),
    ("YOU FOOD WANT", "Do you want food?"),
    ("YOU OK", "Are you okay?"),
    
    # Negation
    ("I KNOW NO", "I don't know."),
    ("I WANT NO", "I don't want."),
    ("HE COME NO", "He doesn't come."),
    ("I UNDERSTAND NO", "I don't understand."),
    
    # Complex sentences
    ("I SCHOOL GO LEARN", "I go to school to learn."),
    ("I FRIEND HELP WANT", "I want to help my friend."),
    ("HE DOCTOR SEE NEED", "He needs to see the doctor."),
    ("I HOME GO WANT", "I want to go home."),
    ("SHE TEACHER GOOD", "She is a good teacher."),
    ("I HAPPY FEEL", "I feel happy."),
    ("HE FRIEND MY", "He is my friend."),
    ("I YOU LOVE", "I love you."),
    ("I YOU THANK", "I thank you."),
    ("WE TOGETHER WORK", "We work together."),
    ("I FAMILY LOVE", "I love my family."),
    ("SHE STUDENT GOOD", "She is a good student."),
    ("I BUS WAIT", "I wait for the bus."),
    ("HE CAR DRIVE", "He drives a car."),
    ("I PHONE CALL", "I make a phone call."),
]

# Write to CSV
CSV_PATH = "gloss_sentence_pairs.csv"
with open(CSV_PATH, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['gloss', 'sentence'])
    for gloss, sentence in SYNTHETIC_PAIRS:
        writer.writerow([gloss, sentence])

print(f"Generated {len(SYNTHETIC_PAIRS)} training pairs -> {CSV_PATH}")
print("\nFirst 10 pairs:")
for g, s in SYNTHETIC_PAIRS[:10]:
    print(f"  {g:30s} -> {s}")
print(f"\n>>> REVIEW and EDIT {CSV_PATH} before proceeding! <<<")
print("You can download it, edit, and re-upload, or edit in Colab's file browser.")

In [ ]:
# ===== Load and Prepare the Gloss-Sentence Dataset =====

import pandas as pd
from torch.utils.data import Dataset as TorchDataset, DataLoader
from transformers import T5Tokenizer

CSV_PATH = "gloss_sentence_pairs.csv"  # or upload your own
df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} gloss-sentence pairs")
print(df.head(10))

# Split: 90% train, 10% val
val_size = max(int(len(df) * 0.1), 1)
train_df = df.iloc[:-val_size].reset_index(drop=True)
val_df = df.iloc[-val_size:].reset_index(drop=True)
print(f"\nTrain: {len(train_df)}, Val: {len(val_df)}")

MODEL_NAME = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME, legacy=False)

class GlossSentenceDataset(TorchDataset):
    def __init__(self, dataframe, tokenizer, max_input_len=64, max_output_len=128):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_input_len = max_input_len
        self.max_output_len = max_output_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        input_text = f"translate gloss to english: {row['gloss']}"
        target_text = row['sentence']

        input_enc = self.tokenizer(
            input_text, max_length=self.max_input_len,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        target_enc = self.tokenizer(
            target_text, max_length=self.max_output_len,
            padding='max_length', truncation=True, return_tensors='pt'
        )

        labels = target_enc.input_ids.squeeze()
        labels[labels == tokenizer.pad_token_id] = -100

        return {
            'input_ids': input_enc.input_ids.squeeze(),
            'attention_mask': input_enc.attention_mask.squeeze(),
            'labels': labels,
        }

train_dataset = GlossSentenceDataset(train_df, tokenizer)
val_dataset = GlossSentenceDataset(val_df, tokenizer)

print(f"\nDatasets created: {len(train_dataset)} train, {len(val_dataset)} val")

In [ ]:
# ===== Fine-Tune T5-small with LoRA =====

from transformers import T5ForConditionalGeneration, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType

# Load base model
base_model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

# Configure LoRA
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,                    # LoRA rank (higher = more capacity)
    lora_alpha=32,           # scaling factor
    lora_dropout=0.1,
    target_modules=["q", "v"],  # attention query and value projections
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# Training arguments
training_args = TrainingArguments(
    output_dir="./t5_lora_output",
    num_train_epochs=30,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=3e-4,
    weight_decay=0.01,
    warmup_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=10,
    fp16=torch.cuda.is_available(),
    report_to="none",
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print("\nStarting LoRA fine-tuning...")
trainer.train()
print("\nTraining complete!")

In [ ]:
# ===== Test the Fine-Tuned Model =====

model.eval()
test_glosses = [
    "HELLO",
    "I SCHOOL GO",
    "HE FOOD EAT PAST",
    "SHE BOOK READ",
    "I WATER DRINK FUTURE",
    "YOU HELP WANT",
    "WE PARK GO TOMORROW",
    "I FRIEND HELP",
    "I YOU LOVE",
]

print("Fine-tuned model predictions:")
print("=" * 60)
for gloss in test_glosses:
    input_text = f"translate gloss to english: {gloss}"
    inputs = tokenizer(input_text, return_tensors='pt', max_length=64,
                       truncation=True, padding=True).to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=128, num_beams=4,
                                  early_stopping=True)
    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"  Gloss: {gloss}")
    print(f"  Pred : {pred}")
    print()

In [ ]:
# ===== Save and Download LoRA Adapter Weights =====

ADAPTER_DIR = "llm_finetuned"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to {ADAPTER_DIR}/")

# List saved files and sizes
import os
for f in os.listdir(ADAPTER_DIR):
    size = os.path.getsize(f"{ADAPTER_DIR}/{f}")
    print(f"  {f}: {size/1024:.1f} KB")

# Zip for download
import shutil
shutil.make_archive('llm_finetuned', 'zip', '.', ADAPTER_DIR)
print(f"\nZipped to llm_finetuned.zip")

from google.colab import files
files.download('llm_finetuned.zip')
print("\nDownload started!")
print("Extract the zip contents into: models/llm_finetuned/ on your local machine")

## Done!

### Next steps on your local machine:

1. Place `classifier.pt` at `models/classifier.pt`
2. Extract `llm_finetuned.zip` contents into `models/llm_finetuned/`
3. Verify: `python -m src.grammar_mapper.llm_mapper`
4. Run the demo: `streamlit run src/demo/app.py`